# STAT 764 · Meeting 15 — Trees

**Tuesday, October 27**

Every model so far has been a **formula**: a weighted sum of the columns, turned into a
probability. A **decision tree** is a different kind of model — a flowchart. It asks one yes/no
question about a patient ("more than one hospital stay in the past year?"), then another, and the
group of past patients the patient lands with gives the answer: the share of them who came back.

Trees matter less for themselves than for what they lead to. A single tree is easy to read and
rarely the best predictor; averaging hundreds of them — Thursday's random forests, and boosting
next week — gives some of the strongest models for tables like ours. Today is how one tree is
built, why it overfits, and why it is unstable.

**The question for today.** How does a tree choose its questions — and can one tree compete with
Meeting 9's logistic regression on the readmission data?

In [ ]:
import os
import pathlib
import sys

here = pathlib.Path.cwd()
found = ([p for p in [here, *here.parents] if (p / "course" / "stat764.py").exists()]
         + [c.parent.parent for c in here.glob("*/course/stat764.py")])
if os.environ.get("STAT764_REPO"):          # your clone, when it is not above you
    found.insert(0, pathlib.Path(os.environ["STAT764_REPO"]))

if found:
    sys.path.insert(0, str(found[0] / "course"))
else:
    import urllib.request
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/DataScienceUWL/stat764-fall2026"
        "/main/course/stat764.py", "stat764.py")
    sys.path.insert(0, ".")
    print("  (no local clone found — pulled the helpers from GitHub)")

In [ ]:
from collections import Counter
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import HTML
from matplotlib.animation import FuncAnimation
from stat764 import diabetes, load

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, roc_auc_score
from sklearn.model_selection import GridSearchCV, GroupKFold, GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree

# The readmission data and split from Meetings 9-12, rebuilt so this notebook stands on its own.
enc = diabetes()
enc = enc[~enc["discharge_disposition_id"].isin([11, 13, 14, 19, 20, 21])]  # death, hospice
y_all = (enc["readmitted"] == "<30").astype(int).to_numpy()

NUM = ["time_in_hospital", "num_lab_procedures", "num_procedures", "num_medications",
       "number_outpatient", "number_emergency", "number_inpatient", "number_diagnoses"]
CAT = ["age", "gender", "race", "admission_type_id", "discharge_disposition_id",
       "admission_source_id", "A1Cresult", "max_glu_serum", "insulin", "change",
       "diabetesMed", "medical_specialty"]
X_all = enc[NUM + CAT].astype({c: str for c in CAT})
patients = enc["patient_nbr"].to_numpy()

# Split by PATIENT, so nobody is on both sides -- the same split as Meetings 9-12.
train, test = next(GroupShuffleSplit(1, test_size=0.25, random_state=764)
                   .split(X_all, y_all, groups=patients))
X_train, y_train, X_test, y_test = X_all.iloc[train], y_all[train], X_all.iloc[test], y_all[test]


def readmission_model(**options):
    """Meeting 9's logistic regression. `options` go to LogisticRegression."""
    return Pipeline([
        ("prep", ColumnTransformer([
            ("num", Pipeline([("fill", SimpleImputer(strategy="median")),
                              ("scale", StandardScaler())]), NUM),
            ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=20), CAT),
        ])),
        ("model", LogisticRegression(max_iter=2000, **options)),
    ])


def tree_model(**options):
    """The same columns, a decision tree at the end. `options` go to DecisionTreeClassifier.
    No scaler: a tree only asks "above or below this value?", so units do not matter -- and the
    split values stay in the columns' own units (stays, days), which makes the tree readable."""
    return Pipeline([
        ("prep", ColumnTransformer([
            ("num", SimpleImputer(strategy="median"), NUM),
            ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=20), CAT),
        ], verbose_feature_names_out=False)),
        ("model", DecisionTreeClassifier(random_state=0, **options)),
    ])


def show_tree(fitted):
    """Print a fitted tree pipeline as indented if/else rules, in the columns' own names."""
    print(export_text(fitted[-1], feature_names=list(fitted[0].get_feature_names_out()),
                      show_weights=True, decimals=1))


def by_decile(probs, outcome):
    """Meeting 11's check: up to ten groups by probability, what the model SAID against what
    HAPPENED. A tree gives only a few different numbers, so some tenths merge."""
    group = pd.qcut(probs, 10, labels=False, duplicates="drop")
    return pd.DataFrame({"said": [probs[group == k].mean() for k in np.unique(group)],
                         "came back": [outcome[group == k].mean() for k in np.unique(group)]},
                        index=pd.Index(np.unique(group) + 1, name="group")).round(3)


print(f"  train {len(train):,} discharges · test {len(test):,} · "
      f"readmitted within 30 days: {y_test.mean():.1%} of test")

## 0. What a decision tree is

A **decision tree** is one of the simplest models for classification. It divides past cases into
groups, one yes/no question at a time, aiming for groups that are more and more **alike in the
outcome**: mostly yes, or mostly no. Drawn out, it is a flowchart. A triage nurse works the same way
(*chest pain? → yes → over 50? → yes → see a doctor now*), except that in a tree model **the computer
chooses the questions** from past cases.

A tree is easy to read as a picture. The interesting statistics are in **how the computer decides
where to split**, which is Section 1. First, the picture: a tree for Mini-project 1's bank calls —
*will this client say yes?* — built from two questions someone might ask first:

```
Did a previous campaign succeed with this client?
├── yes ............................................ leaf A
└── no  → Was the client called on a cell phone?
          ├── yes .................................. leaf B
          └── no  .................................. leaf C
```

- A **split** is one yes/no question. The first one is the **root**.
- A **leaf** is an end point: a group of past calls.
- The **depth** is the number of questions on the longest path — 2 here.

**To classify a new client, follow the branches until you reach a leaf.** Take a client no earlier
campaign won over, called on a landline: *no* at the first question, *no* at the second, so leaf C.
The tree predicts what happened in leaf C: the **share of its past calls that said yes**, 4.9% in
the next cell. (For a number, such as a house price, a leaf predicts its cases' **average**.)

That 4.9% is just a share of past calls. Whether it works as a **probability for future clients** —
do clients told 5% say yes about 5% of the time? — is the separate calibration check from Meetings
11 and 12, the same for a tree as for logistic regression. The next cell counts the three leaves.

In [ ]:
# The bank's 37,069 past calls from Mini-project 1, sorted into the three leaves of the tree above.
calls = load("bank_calls.csv")
said_yes = calls["y"] == "yes"
success = calls["poutcome"] == "success"       # question 1: did a previous campaign succeed?
cell = calls["contact"] == "cellular"          # question 2: was the client called on a cell phone?
leaves = {"A  previous success": success,
          "B  no success, cell phone": ~success & cell,
          "C  no success, landline": ~success & ~cell}

print(f"all calls: {len(calls):,} · {said_yes.mean():.1%} said yes\n")
for name, in_leaf in leaves.items():
    # The leaf's share who said yes IS the tree's prediction for every new client who lands there.
    print(f"  leaf {name:<27} {in_leaf.sum():>6,} calls · {said_yes[in_leaf].mean():.1%} said yes")

**What this shows.** Three leaves, three predictions — 65.7%, 12.2% and 4.9%. Every client who lands
in a leaf gets the same number, so a tree can only give as many different predictions as it has
leaves.

**How the computer builds a tree, in four steps** — the rest of today shows each one:

1. **Try every possible question** on every column — every cut point for a number, every grouping
   for a category.
2. **Keep the one that best separates** the yes cases from the no cases. Section 1 measures "best".
3. **Repeat inside each side**, on that side's cases only.
4. **Stop** when the groups get too small, the tree too deep, or no question helps (Section 4).

## 1. How a tree chooses a question

Today's data is Meeting 9's: hospital patients with diabetes, and whether each **came back** within
30 days. Start with all the patients in one group. The computer wants to divide them in two so that
each side is more alike — mostly came back, or mostly did not. For each candidate question it asks:

> **How much would this question reduce the impurity of the groups?**

**Impurity.** In a group where a share *p* came back, the **Gini impurity** is
1 − *p*² − (1 − *p*)², which for two outcomes is the same as **2*p*(1 − *p*)**.

- If *p* = 0 or *p* = 1 — no one came back, or everyone did — the Gini is **0**: the group is **pure**.
- If *p* = 0.5 the Gini is **0.5**, its largest: the group is as mixed as it can be.

The next cell writes the two formulas as functions, then draws the curve.

In [ ]:
def gini(came_back, total):
    """Gini impurity of a group: 1 - p^2 - (1 - p)^2, the same as 2p(1 - p), with p = came_back / total."""
    p = came_back / total
    return 1 - p ** 2 - (1 - p) ** 2


def split_gini(left_back, left_total, right_back, right_total):
    """The impurity after a split: each side's Gini, weighted by its share of the patients."""
    n = left_total + right_total
    return (left_total / n) * gini(left_back, left_total) + (right_total / n) * gini(right_back, right_total)


# The Gini curve: impurity against the share who came back. Zero at both ends, highest in the middle.
shares = np.linspace(0, 1, 201)
fig, ax = plt.subplots(figsize=(6.5, 3.2))
ax.plot(shares, 2 * shares * (1 - shares), color="#1f3f7a", lw=2.5)
for s, offset in [(0.1, (8, -14)), (0.3, (8, -14)), (0.5, (10, 4))]:   # points the example below uses
    ax.plot(s, 2 * s * (1 - s), "o", color="#8a3a12", ms=7)
    ax.annotate(f"p = {s}: Gini {2 * s * (1 - s):.2f}", (s, 2 * s * (1 - s)), xytext=offset,
                textcoords="offset points", fontsize=10, color="#8a3a12")
ax.set(xlabel="share of the group who came back, p", ylabel="Gini impurity, 2p(1 − p)",
       ylim=(0, 0.56), title="Impurity: 0 for a pure group, largest at half and half")
plt.show()

### Scoring one question, by hand first

Made-up round numbers first, so the arithmetic is easy to follow: **100 patients, 30 of whom came
back.** Before any question, *p* = 0.3, so

> Gini before = 2 × 0.3 × 0.7 = **0.42**.

Candidate **question A**, *in the hospital more than a week?*, sends the patients two ways:

| side | came back | did not | patients | share who came back | Gini |
|---|---|---|---|---|---|
| yes, more than a week | 18 | 2 | 20 | 0.9 | 2 × 0.9 × 0.1 = 0.18 |
| no | 12 | 68 | 80 | 0.15 | 2 × 0.15 × 0.85 = 0.255 |

Both sides are purer than the 0.42 we started with. To score the question, average the two sides'
Gini, **each weighted by its share of the patients**:

> Gini after = (20/100) × 0.18 + (80/100) × 0.255 = 0.036 + 0.204 = **0.240**
>
> **Reduction** = 0.42 − 0.240 = **0.180**

The computer scores **every** candidate question this way and keeps the one with the **largest
reduction**. The next cell checks question A with the two functions and scores two more.

In [ ]:
# Each candidate: (came back, patients) on the "yes" side, then the same on the "no" side.
# Made-up counts -- 100 patients, 30 came back. Try it: change a count and rerun.
candidates = {
    "A  in the hospital more than a week?":  (18, 20, 12, 80),
    "B  more than 15 medications?":          (10, 50, 20, 50),
    "C  five or more emergency visits?":     (5, 5, 25, 95),
}
before = gini(30, 100)
print(f"before any question: 100 patients, 30 came back -> Gini {before:.3f}\n")
for question, (yes_back, yes_n, no_back, no_n) in candidates.items():
    after = split_gini(yes_back, yes_n, no_back, no_n)   # the size-weighted Gini of the two sides
    print(f"  {question:38} yes: {yes_back}/{yes_n} (Gini {gini(yes_back, yes_n):.3f}) · "
          f"no: {no_back}/{no_n} (Gini {gini(no_back, no_n):.3f}) -> after {after:.3f}, "
          f"reduction {before - after:.3f}")

**What this shows.** Question A wins: it lowers the impurity by **0.180**. Question B barely helps
(0.020): each side is still a mix. Question C finds a group that is **perfectly pure** — all 5
came back, Gini 0 — but it sorts only 5 of the 100 patients, so its reduction is only **0.052**.
That is why each side is **weighted by its size**: a pure group earns credit in proportion to how
many patients it holds.

The best any question could do here is a reduction of 0.42: all 30 who came back on one side,
everyone else on the other. A real tree can only use the splits its columns allow.

### Columns with numbers: every cut

A column such as *prior hospital stays* holds many possible questions: *at most 0? at most 1? at
most 2? …* The computer sorts the values that occur and tries a cut **halfway between each
neighboring pair** (0.5, 1.5, 2.5, …). Each cut makes a different pair of groups, scored exactly as
above. It runs that search on **every** column, keeps the single best question overall, and may
ask about the same column again further down the tree.

Every candidate starts from the same group, so the **lowest weighted Gini** and the **largest
reduction** pick the same question. The pictures below plot the weighted Gini. To watch the search,
take 1,000 real training discharges at random and one column: the number of hospital stays in the
past year.

In [ ]:
# 1,000 training discharges drawn at random -- a fixed seed, so the pictures are the same every run.
picked = np.random.default_rng(55).choice(len(train), 1_000, replace=False)
small_X, small_y = X_train.iloc[picked], y_train[picked]
stays = small_X["number_inpatient"].to_numpy()           # the one column: prior hospital stays

# The candidate cuts sit halfway between neighboring values that occur: 0.5, 1.5, 2.5, ...
values = np.unique(stays)
cuts = (values[:-1] + values[1:]) / 2

# Score every cut: each side's size, share who came back and Gini, then the size-weighted total.
scored = []
for cut in cuts:
    left = stays <= cut                                   # "at most the cut?" -- yes goes left
    scored.append({"at most": cut,
                   "left n": left.sum(), "left share": small_y[left].mean(),
                   "left Gini": gini(small_y[left].sum(), left.sum()),
                   "right n": (~left).sum(), "right share": small_y[~left].mean(),
                   "right Gini": gini(small_y[~left].sum(), (~left).sum()),
                   "weighted Gini": split_gini(small_y[left].sum(), left.sum(),
                                               small_y[~left].sum(), (~left).sum())})
scored = pd.DataFrame(scored)
before_gini = gini(small_y.sum(), len(small_y))
print(f"before any question: {len(small_y):,} discharges, {small_y.mean():.1%} came back, "
      f"Gini {before_gini:.4f}")
print(scored.round(4).to_string(index=False))

In [ ]:
# The search, animated: one frame per cut, then a last frame for the winner. Press play.
# Left: each bar is the share who came back among the patients with that many prior stays (under
# it, in brackets, how many patients). The dashed line is the cut; the thick flat lines are the
# share who came back on each side of it. Right: the weighted Gini of every cut tried so far.
pos = np.arange(len(values))                     # bar k is values[k]; cut k sits at k + 0.5
share = [small_y[stays == v].mean() for v in values]
count = [(stays == v).sum() for v in values]
best = scored["weighted Gini"].idxmin()
low, high = scored["weighted Gini"].min(), max(scored["weighted Gini"].max(), before_gini)
BLUE, ORANGE = "#1f3f7a", "#8a3a12"              # the "at most" side and the "more" side

fig, (ax_bars, ax_curve) = plt.subplots(1, 2, figsize=(12, 4.8))
fig.subplots_adjust(left=0.06, right=0.98, bottom=0.2, top=0.92, wspace=0.22)


def draw(frame):
    """Draw cut number `frame` -- or, on the last frame, the best cut."""
    k = best if frame == len(cuts) else frame
    row = scored.iloc[k]
    ax_bars.clear()
    ax_curve.clear()

    # Left panel: the two sides shaded, the bars, and each side's share who came back.
    ax_bars.axvspan(-0.5, k + 0.5, color="#4c72b0", alpha=0.15)
    ax_bars.axvspan(k + 0.5, len(values) - 0.5, color="#dd8452", alpha=0.18)
    ax_bars.bar(pos, share, width=0.7, color="#8c8c8c")
    ax_bars.hlines(row["left share"], -0.5, k + 0.5, color=BLUE, lw=3)
    ax_bars.hlines(row["right share"], k + 0.5, len(values) - 0.5, color=ORANGE, lw=3)
    ax_bars.axvline(k + 0.5, color="black", ls="--")
    ax_bars.text(0.02, 0.97, f"at most {row['at most']:g}: {row['left n']:.0f} patients, "
                 f"{row['left share']:.1%} came back, Gini {row['left Gini']:.4f}",
                 transform=ax_bars.transAxes, va="top", color=BLUE, fontsize=10,
                 bbox=dict(fc="white", ec="none", alpha=0.85))
    ax_bars.text(0.02, 0.89, f"more than {row['at most']:g}: {row['right n']:.0f} patients, "
                 f"{row['right share']:.1%} came back, Gini {row['right Gini']:.4f}",
                 transform=ax_bars.transAxes, va="top", color=ORANGE, fontsize=10,
                 bbox=dict(fc="white", ec="none", alpha=0.85))
    ax_bars.set(xticks=pos, xticklabels=[f"{v}\n({n})" for v, n in zip(values, count)],
                xlim=(-0.5, len(values) - 0.5), ylim=(0, 0.8),
                xlabel="prior hospital stays (patients)", ylabel="share who came back")
    ax_bars.set_title(f"Question: prior stays at most {row['at most']:g}?", loc="left")

    # Right panel: every cut tried so far, the current one circled. Dotted: before any question.
    tried = scored if frame == len(cuts) else scored.iloc[:frame + 1]
    ax_curve.plot(tried["at most"], tried["weighted Gini"], "o-", color="#4d4d4d")
    ax_curve.plot(row["at most"], row["weighted Gini"], "o", ms=16, mfc="none", mec="#c00000", mew=2.5)
    ax_curve.axhline(before_gini, color="#6e6e6e", ls=":")
    ax_curve.text(cuts[-1] + 0.4, before_gini, "before any question", ha="right", va="bottom", fontsize=9)
    ax_curve.text(0.02, 0.04, f"{row['left n']:.0f}/1,000 × {row['left Gini']:.4f} + "
                  f"{row['right n']:.0f}/1,000 × {row['right Gini']:.4f} = {row['weighted Gini']:.4f}",
                  transform=ax_curve.transAxes, fontsize=10)
    ax_curve.set(xticks=cuts, xlim=(cuts[0] - 0.5, cuts[-1] + 0.5),
                 ylim=(low - 0.0015, high + 0.0012),
                 xlabel="cut: prior stays at most …", ylabel="weighted Gini (lower is better)")
    if frame == len(cuts):
        ax_curve.set_title(f"Lowest of all {len(cuts)}: at most {row['at most']:g} stays", loc="left",
                           color="#c00000", fontweight="bold")
    else:
        ax_curve.set_title(f"Cut {frame + 1} of {len(cuts)}", loc="left")


anim = FuncAnimation(fig, draw, frames=len(cuts) + 1, interval=1500)
plt.close(fig)                                   # show only the animation, not a stray still
HTML(anim.to_jshtml())

In [ ]:
# The same search as one still picture: every cut's weighted Gini, the lowest starred.
fig, ax = plt.subplots(figsize=(7.5, 3.8))
ax.plot(scored["at most"], scored["weighted Gini"], "o-", color="#4d4d4d")
ax.plot(scored["at most"][best], scored["weighted Gini"][best], "*", ms=20, color="#c00000")
ax.annotate(f"lowest: at most {scored['at most'][best]:g} stays, {scored['weighted Gini'][best]:.4f}",
            (scored["at most"][best], scored["weighted Gini"][best]), xytext=(12, -4),
            textcoords="offset points", color="#c00000", fontsize=10)
ax.axhline(before_gini, color="#6e6e6e", ls=":")
ax.text(cuts[-1], before_gini, f"before any question, {before_gini:.4f}", ha="right", va="bottom",
        fontsize=9)
ax.set(xticks=cuts, xlabel="cut: prior stays at most …", ylabel="weighted Gini (lower is better)",
       title="Weighted Gini against the cut point, 1,000 discharges")
plt.show()

# The picture in words: the winner, and the closest runner-up.
second = scored["weighted Gini"].nsmallest(2).index[1]
print(f"lowest: at most {scored['at most'][best]:g} stays -- {scored['left n'][best]} patients, "
      f"{scored['left share'][best]:.1%} came back, against {scored['right n'][best]}, "
      f"{scored['right share'][best]:.1%} -- weighted Gini {scored['weighted Gini'][best]:.4f}, "
      f"a reduction of {before_gini - scored['weighted Gini'][best]:.4f}")
print(f"runner-up: at most {scored['at most'][second]:g} stays -- {scored['right n'][second]} patients "
      f"above it, {scored['right share'][second]:.1%} came back -- weighted Gini "
      f"{scored['weighted Gini'][second]:.4f}, a reduction of {before_gini - scored['weighted Gini'][second]:.4f}\n")

# A tree runs that search on EVERY column, then keeps the overall lowest. The best cut on each count:
by_column = []
for col in NUM:
    x = small_X[col].to_numpy()
    candidates = (np.unique(x)[:-1] + np.unique(x)[1:]) / 2
    score = {c: split_gini(small_y[x <= c].sum(), (x <= c).sum(), small_y[x > c].sum(), (x > c).sum())
             for c in candidates}
    winner = min(score, key=score.get)
    by_column.append({"column": col, "cuts tried": len(candidates), "best cut: at most": winner,
                      "weighted Gini": score[winner]})
print(pd.DataFrame(by_column).sort_values("weighted Gini").round(4).to_string(index=False))

**What this shows.** Before any question, 12.1% of the 1,000 came back: Gini 0.2127. The lowest
weighted Gini is at the cut **at most 1.5** — 0 or 1 stay against 2 or more: 835 patients of whom
9.2% came back, against 165 of whom 26.7% did. Its score is 835/1,000 × 0.1674 + 165/1,000 ×
0.3911 = **0.2043**, a reduction of only 0.0084. Real reductions are small when few patients come
back; what matters is which is largest. The cut at 4.5 finds a far riskier group — 46.9% of its 32
patients came back — and almost wins (0.2047), but it sorts out only 32 patients: question C's
lesson again, each side counts by its size. Every column gets the same search, and of the eight
counts, prior stays has the lowest score.

### Then the same search inside each answer

The winning question splits the patients in two. The tree then runs the **same search inside each
side, separately**, and keeps going — that is what *recursive* means. It is also *greedy*: each
question is the best one **right now**, and an earlier question is **never revisited**, even if a
different first question would have led to better questions later.

Below, the same 1,000 patients on two columns, prior stays across and days in the hospital up. Each
shaded box is a group, labeled with its share who came back and its size, after one, two and three
splits. Then the same tree as a flowchart.

In [ ]:
# The same 1,000 patients on two columns. A tree two questions deep makes three splits: the first
# question, then one inside each answer.
two = ["number_inpatient", "time_in_hospital"]
small_tree = DecisionTreeClassifier(max_depth=2, random_state=0).fit(small_X[two], small_y)
t = small_tree.tree_                             # the fitted tree's arrays: feature, threshold, ...
splits = [node for node in range(t.node_count) if t.children_left[node] != -1]   # in the order made


def groups(node, rows, box, made):
    """The groups once the splits in `made` are done: each group's box on the picture
    (x from, x to, y from, y to) and which patients are in it (True/False for each)."""
    if node not in made:
        return [(box, rows)]
    col, cut = t.feature[node], t.threshold[node]
    yes = rows & (small_X[two[col]].to_numpy() <= cut)    # "at most the cut?" -- yes goes left
    x0, x1, y0, y1 = box
    if col == 0:                                          # a cut on prior stays: a vertical line
        yes_box, no_box = (x0, cut, y0, y1), (cut, x1, y0, y1)
    else:                                                 # a cut on days: a horizontal line
        yes_box, no_box = (x0, x1, y0, cut), (x0, x1, cut, y1)
    return (groups(t.children_left[node], yes, yes_box, made)
            + groups(t.children_right[node], rows & ~yes, no_box, made))


# Jitter the dots a little, so patients with the same two numbers don't sit on top of each other.
jitter = np.random.default_rng(0).uniform(-0.3, 0.3, size=(len(small_y), 2))
dot_x = small_X[two[0]].to_numpy() + jitter[:, 0]
dot_y = small_X[two[1]].to_numpy() + jitter[:, 1]
outer = (-0.5, small_X[two[0]].max() + 0.5, 0.5, small_X[two[1]].max() + 0.5)

fig, axes = plt.subplots(1, 3, figsize=(14, 4.8), sharey=True)
for n_splits, ax in zip([1, 2, 3], axes):
    for (x0, x1, y0, y1), rows in groups(0, np.ones(len(small_y), bool), outer, splits[:n_splits]):
        back = small_y[rows].mean()
        ax.add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, facecolor=plt.cm.Oranges(back / 0.6),
                                   edgecolor="black", lw=1.5))
        ax.text((x0 + x1) / 2, (y0 + y1) / 2, f"{back:.0%}\nof {rows.sum()}", ha="center",
                va="center", fontsize=10, bbox=dict(boxstyle="round", fc="white", ec="none"))
    ax.scatter(dot_x[small_y == 0], dot_y[small_y == 0], s=3, color="#5a5a5a", alpha=0.5)
    ax.scatter(dot_x[small_y == 1], dot_y[small_y == 1], s=14, color="black", marker="x", lw=0.8)
    newest = splits[n_splits - 1]
    ax.set(xlim=outer[:2], ylim=outer[2:], xlabel="prior hospital stays",
           title=f"after {n_splits} split{'s' if n_splits > 1 else ''}\n"
                 f"new: {two[t.feature[newest]]} ≤ {t.threshold[newest]:g}")
axes[0].set_ylabel("days in the hospital")
fig.suptitle("Each box: share who came back, of how many patients  (× came back · dot did not)",
             fontsize=11)
fig.tight_layout()
plt.show()

# The same tree as a flowchart: each box is a group, with its question (if it asks one) and its Gini.
# Left is "yes" (True), right is "no" (False). Shaded like the pictures above: darker, more came back.
fig, ax = plt.subplots(figsize=(12, 4.8))
for box in plot_tree(small_tree, feature_names=two, rounded=True, fontsize=10, ax=ax):
    # scikit-learn writes "value = [did not, came back]"; say it in words, with the share.
    counts = re.search(r"value = \[([\d.]+), ([\d.]+)\]", box.get_text())
    if counts:                                   # (the "True"/"False" arrow labels have none)
        did_not, came = map(float, counts.groups())
        box.set_text(re.sub(r"samples = .*\nvalue = .*",
                            f"{did_not + came:.0f} patients, {came:.0f} came back\n"
                            f"share {came / (did_not + came):.3f}", box.get_text()))
        box.get_bbox_patch().set_facecolor(plt.cm.Oranges(came / (did_not + came) / 0.6))
plt.show()

**What this shows.** The first question splits the 1,000 into 835 (9% came back) and 165 (27%).
Then the tree searches **inside each side on its own**: among the 835 the best question is about
days in the hospital (at most 4.5: 7% of 530 came back; more: 13% of 305), and among the 165 it
is prior stays **again**, at a new cut (at most 4.5: 22% of 133; more: 47% of 32). The first cut
never moves — each new line divides one box that is already there.

### Down the tree: a prediction

To predict, a patient answers the questions from the top until there are none left. The group it
lands in is a **leaf**, and the tree's probability is the share of that leaf's **training**
patients who came back.

In [ ]:
# One new patient -- from the test set, so this tree never saw them -- answers the questions from the
# top. We take the first test patient with 3 prior stays.
patient = X_test[X_test["number_inpatient"] == 3].iloc[[0]]
node = 0
while t.children_left[node] != -1:               # -1: no children, so a leaf -- no more questions
    col, cut = two[t.feature[node]], t.threshold[node]
    value = patient[col].iloc[0]
    print(f"  {col} = {value}: at most {cut:g}? {'yes, go left' if value <= cut else 'no, go right'}")
    node = t.children_left[node] if value <= cut else t.children_right[node]

# The leaf's training patients: the ones among the 1,000 that ended up in the same group.
same_group = small_tree.apply(small_X[two]) == node
print(f"  its leaf: {same_group.sum()} training patients, {small_y[same_group].sum()} came back "
      f"-> share {small_y[same_group].mean():.3f}")
print(f"  the tree's probability for this patient: {small_tree.predict_proba(patient[two])[0, 1]:.3f}")

# Every patient, old or new, lands in one of the leaves -- so gets one of only that many numbers.
every = np.unique(small_tree.predict_proba(small_X[two])[:, 1])
print(f"  different probabilities this tree can give: {len(every)} -> {every.round(3)}")

**What this shows.** This patient had 3 prior stays: *no* at the first question (more than 1.5),
then *yes* at the second (at most 4.5) — the same column asked twice, at a different cut. They
land in a leaf of 133 training patients, 29 of whom came back, so the tree says **0.218**: exactly
that share. Every patient who lands in that leaf gets 0.218, whatever else is true of them, so
this tree can say only **four** different numbers. Section 4's tuned tree has more leaves, but the
same limit.

## 2. On all the patients: the tree's first split

On the full training set the computer runs the same search over **every** column — the cut points
of each count, and each category of the other columns (an emergency admission or not, age 70–80 or
not, …) — on all 74,637 discharges, and keeps the single question with the lowest weighted Gini. A
tree with just one question is called a **stump**.

In [ ]:
# The tree the computer grows with ONE question (a "stump"), on all 74,637 training discharges.
stump = tree_model(max_depth=1).fit(X_train, y_train)
show_tree(stump)

# The same split, counted by hand: patients with at most 1 hospital stay in the past year, and the rest.
few = X_train["number_inpatient"].to_numpy() <= 1
before = gini(y_train.sum(), len(y_train))
after = split_gini(y_train[few].sum(), few.sum(), y_train[~few].sum(), (~few).sum())
print(f"at most 1 stay: {few.sum():,} discharges, {y_train[few].mean():.1%} came back")
print(f"2 or more:      {(~few).sum():,} discharges, {y_train[~few].mean():.1%} came back")
print(f"Gini before the split {before:.4f} · after {after:.4f}")

**What this shows.** Of all the questions it could ask, the tree's first is about **prior hospital
stays** — the same cut Section 1's 1,000 patients chose: patients with two or more stays in the past
year come back more than twice as often (22.5% against 9.7%). The weights in brackets are the counts
of patients who did not and did come back in each group. The impurity drops only a little —
readmission is hard to predict — but no other single question drops it more.

### How the tree asks about a category

scikit-learn's trees ask only one kind of question: *is this number at most a cut?* So inside
`tree_model`, a `OneHotEncoder` first turns each category column into **one 0/1 column per
category**. A question about a category becomes "is this 0/1 column at most 0.5?", which in words
is "is this patient *not* in that category?". For example, `discharge_disposition_id_22 <= 0.5`
sends the patients who were **not** sent to a rehab facility one way, and the rest the other.

Four things to know when you read a tree's questions:

- **Codes are categories, not amounts.** Admission type, discharge disposition and admission source
  are stored as code numbers, but code 22 is not "more" than code 3. The setup cell turns these
  columns into text (`astype(str)`), so the tree never cuts them the way it cuts counts.
- **Missing is its own category.** A blank becomes the category `nan`: `A1Cresult_nan` is 1 when no
  A1C result was recorded. Not being tested can itself say something about a patient.
- **Rare categories are pooled.** A category with fewer than 20 training stays shares one column
  with the other rare ones, named `..._infrequent_sklearn` (`min_frequency=20`). A category the
  training stays never had is 0 in every column (`handle_unknown="ignore"`).
- **One category against the rest.** Each question singles out one category. To separate several
  categories from the others, the tree needs several questions in a row. (Some tree software, such
  as R's `rpart`, can split a column's categories into any two groups with one question;
  scikit-learn's trees cannot.)

The next cell counts the columns the encoder makes.

In [ ]:
# What the encoder inside tree_model makes of the 12 category columns: one 0/1 column per category.
encoder = tree_model()[0].fit(X_train)            # the pipeline's first step, fit on the training stays
encoded_names = list(encoder.get_feature_names_out())
print(f"{len(NUM)} count columns + {len(encoded_names) - len(NUM)} category columns = "
      f"{len(encoded_names)} columns the tree can ask about\n")
for col in ["discharge_disposition_id", "A1Cresult", "medical_specialty"]:
    made = [n for n in encoded_names if n.startswith(col + "_")]
    # dropna=False counts "not recorded" as a category, the way the encoder does
    print(f"  {col}: {X_train[col].nunique(dropna=False)} categories in the data -> {len(made)} columns, "
          f"e.g. {', '.join(made[:3])}{', ...' if len(made) > 3 else ''}")
    if any("infrequent" in n for n in made):
        print(f"      the rare ones share {col}_infrequent_sklearn")

**What this shows.** The 12 category columns become 112 yes/no columns, so every search chooses among
120 columns: the 8 counts and the 112 category columns. Discharge disposition has 21 codes in the
training stays but only 16 columns, because its rarest codes share one. Medical specialty's 72 values,
counting "not recorded", shrink to 43. In question 3 of the studio, names like these are the
questions you will be tallying.

## 3. Grow it — and watch it overfit

Each answer becomes a new group, and the tree asks the next best question inside it. **Depth** is
how many questions in a row. Two questions deep, the tree is still readable:

In [ ]:
show_tree(tree_model(max_depth=2).fit(X_train, y_train))

(*discharge_disposition_id_22* is 1 when the patient was sent on to a rehabilitation facility — code
22 in the dataset's code list.)

Deeper trees can carve the patients into smaller and smaller groups. Here is what that does to the
AUC on the patients the tree was **fitted on** and on the **test** patients it never saw:

In [ ]:
# The same tree, grown deeper and deeper (about 10 seconds).
rows = []
for depth in [1, 2, 4, 6, 8, 12, 20]:
    fitted = tree_model(max_depth=depth).fit(X_train, y_train)
    rows.append({"depth": depth, "groups at the bottom": fitted[-1].get_n_leaves(),
                 "AUC on its own patients": roc_auc_score(y_train, fitted.predict_proba(X_train)[:, 1]),
                 "AUC on test": roc_auc_score(y_test, fitted.predict_proba(X_test)[:, 1])})
print(pd.DataFrame(rows).round(3).to_string(index=False))

In [ ]:
# All the way down: no limit on depth, so it keeps splitting until every group is pure.
full = tree_model().fit(X_train, y_train)
print(f"grown all the way: {full[-1].get_n_leaves():,} groups at the bottom · "
      f"AUC on its own patients {roc_auc_score(y_train, full.predict_proba(X_train)[:, 1]):.3f} · "
      f"on test {roc_auc_score(y_test, full.predict_proba(X_test)[:, 1]):.3f}")

**What this shows.** On its own patients the AUC climbs with every extra question, to a perfect
1.000 when the tree is grown all the way: each of its thousands of tiny groups is pure, so it has
memorized who came back. On new patients the AUC rises for the first few questions, then **falls**,
and the fully grown tree is barely better than a coin. Meeting 3's lesson, in its starkest form: a
flexible enough model fits noise.

## 4. Stop it early — tuned by cross-validation

Two ways to keep a tree honest. **Stop it early:** limit the depth (`max_depth`), or refuse to
make a group smaller than some number of patients (`min_samples_leaf`). **Prune it back:** grow it
fully, then cut off the branches that add the least (`ccp_alpha`, ISLP §8.1.1). Either way the
setting is chosen the way we choose any setting: **by cross-validation on the training patients**,
inside the pipeline — and here split **by patient**, so the folds never share a person (Meeting
9's rule).

In [ ]:
# Choose the depth by 5-fold cross-validation, folds split by patient (about 8 seconds).
search = GridSearchCV(tree_model(), {"model__max_depth": [3, 4, 5, 6, 8]},
                      cv=GroupKFold(5), scoring="roc_auc", n_jobs=-1)
search.fit(X_train, y_train, groups=patients[train])
best_tree = search.best_estimator_
print("cross-validated AUC by depth:",
      dict(zip([3, 4, 5, 6, 8], search.cv_results_["mean_test_score"].round(4))))
print(f"chosen depth: {search.best_params_['model__max_depth']}")

# One honest comparison on the test patients: the tuned tree against Meeting 9's model.
logistic = readmission_model().fit(X_train, y_train)
p_tree = best_tree.predict_proba(X_test)[:, 1]
p_logit = logistic.predict_proba(X_test)[:, 1]
for name, p in [("tuned tree", p_tree), ("Meeting 9's logistic regression", p_logit)]:
    print(f"  {name:32} AUC {roc_auc_score(y_test, p):.4f} · Brier {brier_score_loss(y_test, p):.4f}"
          f" · {len(np.unique(p)):,} different probabilities")

**What this shows.** The best tree, honestly chosen, still ranks the test patients **worse than the
plain logistic regression**, and its probabilities are **coarser**: every patient in the same final
group gets the same number — that group's share who came back — so 24,706 patients share a few
dozen probabilities. That is why the calendar calls a single tree "a stepping stone to ensembles,
not a working method".

## 5. Why a tree is a stepping stone

Trees are **unstable**: change the training patients a little, and the questions change. Fit the
same two-question tree on two random samples of 1,000 training discharges:

In [ ]:
rng = np.random.default_rng(15)
for draw in (1, 2):
    sample = rng.choice(len(train), 1_000, replace=False)
    print(f"sample {draw}:")
    show_tree(tree_model(max_depth=2).fit(X_train.iloc[sample], y_train[sample]))

**What this shows.** Same kind of patients, same method, a different tree — and a reader would tell
a different story from each. A tree's predictions wobble the same way. That weakness is exactly
what Thursday exploits: grow **many** trees on different samples and **average** them. The wobbles
cancel; the signal stays.

**One tree, in summary:**

| trees are good at | a single tree is weak at |
|---|---|
| reading like a flowchart you can explain to a manager | **instability:** a slightly different sample can change the first question, and everything below it |
| finding cut points and combinations ("old **and** many prior stays") with no straight line assumed | **coarse predictions:** only as many values as it has leaves |
| not caring about units — no scaling needed | **memorizing** the training cases when grown deep |
| | **greedy:** each question is the best one *right now*, never revisited |

## Studio

**Everything you need is in this notebook.** Sections 0–5 above explain each idea, and each question
below lists its steps. Work through them in order, at your own pace, alone or with a neighbor.

| | what you practice | about |
|---|---|---|
| **Question 1** | scoring questions by hand, the way a tree does | 10 minutes |
| **Question 2** | tuning a tree honestly, then comparing it with Meeting 9's model | 15 minutes |
| **Question 3** | how much one tree's story changes from sample to sample | 10 minutes |

**All three questions count.** If you do not finish in class, finish at home: the notebook is due 24
hours after class.

**How to work.**

1. If you opened this notebook fresh, run everything above first: **Run All**, or run the cells
   from the top. Questions 1 and 2 use `split_gini`, `tree_model` and `p_logit` from Sections 1–4.
2. Run the **studio setup cell** below the questions. It makes the ten patients (`hand`) and a
   helper for question 3.
3. Before each question, **write your guess** as a comment in its cell, and keep it when the
   answer disagrees. The Compare uses it.
4. Each question ends with one sentence to answer. Write it in a comment or a markdown cell under
   its output.
5. Save and upload to Canvas under "Studio notebook — Day 15" by **Wednesday, October 28, 3:40 pm**.

**Stuck?**

| you see | what to do |
|---|---|
| `NameError` for `hand` or `first_questions` | Run the studio setup cell below the questions. |
| `NameError` for `split_gini`, `tree_model`, `p_logit` or `X_train` | Run the notebook from the top (Run All). |
| `ValueError: The 'groups' parameter should not be None` | Put `groups=patients[train]` inside `.fit(...)`. |
| Question 1: the computer's tree splits on `came_back` | `came_back` is the answer, not a question. X is the three question columns only. |
| Question 3: all twenty trees are identical, second questions too | Make `rng` once, **before** the loop, not inside it. |

### 1. Build a tree by hand — before the computer does

Ten patients — illustrative, not from the data. *Prior stays* is the number of hospital stays in
the past year; *emergency* and *age 70+* are 1 for yes.

| patient | prior stays | emergency | age 70+ | came back |
|---|---|---|---|---|
| 1 | 0 | 0 | 0 | 0 |
| 2 | 0 | 1 | 0 | 0 |
| 3 | 0 | 0 | 1 | 0 |
| 4 | 0 | 1 | 1 | 0 |
| 5 | 1 | 1 | 0 | 0 |
| 6 | 1 | 0 | 1 | 1 |
| 7 | 1 | 1 | 1 | 0 |
| 8 | 2 | 1 | 0 | 1 |
| 9 | 3 | 0 | 1 | 1 |
| 10 | 4 | 1 | 1 | 1 |

**The four candidate first questions.** Each one splits the ten patients into a **yes** side and a
**no** side:

- *prior stays at least 1*: yes for 1 or more prior stays, no for 0.
- *prior stays at least 2*: yes for 2 or more.
- *emergency*: yes when `emergency` is 1.
- *age 70+*: yes when `age_70_plus` is 1.

**A worked example, on a question that is not one of the four: *prior stays at least 3?*** The yes
side is patients 9 and 10: 2 patients, and both came back. The no side is patients 1–8: 8 patients,
2 of whom came back (patients 6 and 8). Its score:

```python
split_gini(2, 8, 2, 2)   # came back on one side, its size, came back on the other side, its size
```

That gives **0.300**: the no side's Gini, 2 × 0.25 × 0.75 = 0.375, weighted by 8/10, plus the yes
side's Gini, 0 (everyone came back), weighted by 2/10. Before any question, 4 of the 10 came back:
Gini 2 × 0.4 × 0.6 = 0.48. The order of the two sides does not change the score.

- **(a)** For each of the four questions, count the patients on each side and how many of them came
  back. Copy this table into a markdown cell and fill it in:

  | question | no side: patients | no side: came back | yes side: patients | yes side: came back |
  |---|---|---|---|---|
  | prior stays at least 1 | | | | |
  | prior stays at least 2 | | | | |
  | emergency | | | | |
  | age 70+ | | | | |

- **(b)** Score each question with `split_gini(...)`, one line per question in the studio cell,
  exactly as in the example. **Lower is better.** Which question would a tree ask first?
- **(c)** Check against the computer. Fit `DecisionTreeClassifier(max_depth=1)` to the table `hand`
  (made for you in the setup cell below) with `hand[["prior_stays", "emergency", "age_70_plus"]]` as X
  and `hand["came_back"]` as y, and print it with `export_text(..., feature_names=[...])`:

  ```python
  cols = ["prior_stays", "emergency", "age_70_plus"]
  machine = DecisionTreeClassifier(max_depth=1, random_state=0).fit(hand[cols], hand["came_back"])
  print(export_text(machine, feature_names=cols, show_weights=True))
  ```

  *Reading it:* a line such as `|--- age_70_plus <= 0.50` means "the patients with `age_70_plus` 0
  go down this branch". The `weights: [a, b]` under a branch mean *a* of its patients did not come
  back and *b* did.

**Answer in a sentence:** did the computer ask your question — and in words, what does that first
split say about these patients?

### 2. A tuned tree against Meeting 9's model

Section 4 chose the tree's **depth** by cross-validation. Here you choose the other stopping rule,
`min_samples_leaf`, the smallest group the tree is allowed to make. Then you compare the tuned tree
with Meeting 9's logistic regression on the test patients.

- **(a)** Tune `min_samples_leaf` over `[100, 200, 400, 800, 1600]` with `GridSearchCV`, 5-fold
  `GroupKFold`, `scoring="roc_auc"`, and `groups=patients[train]` — exactly as Section 4 does for
  the depth. Step by step:
  1. Copy the top of Section 4's search cell into the studio cell: from `search = GridSearchCV(`
     down to the line that prints the chosen depth. Leave out the `best_tree = ...` line.
  2. Rename `search` to `leaf_search` everywhere in your copy, so Section 4's result is not
     overwritten.
  3. Replace the grid `{"model__max_depth": [3, 4, 5, 6, 8]}` with
     `{"model__min_samples_leaf": [100, 200, 400, 800, 1600]}`. Put the same five numbers in the
     first `print`, and ask the second for `'model__min_samples_leaf'`.
  4. Keep `cv=GroupKFold(5)`, `scoring="roc_auc"` and `groups=patients[train]`. The folds are split
     **by patient**, so no patient is both in the data a fold is fit on and in the data that scores it.
  5. Run it (under a minute). It prints the cross-validated AUC for each value and the value chosen.
- **(b)** The honest comparison, on the test patients. `p_logit`, Meeting 9's model's probabilities,
  already exists from Section 4. Make the tuned tree's:

  ```python
  p_leaf = leaf_search.best_estimator_.predict_proba(X_test)[:, 1]
  ```

  Then, for each of `p_leaf` and `p_logit`, print the AUC (`roc_auc_score(y_test, p)`), the Brier
  score (`brier_score_loss(y_test, p)`), the number of different probabilities
  (`len(np.unique(p))`), and the calibration table `by_decile(p, y_test)`. Section 4's `for` loop
  prints the first three; add the fourth.

  *Reading `by_decile`:* each row is a group of test patients with similar predictions. **said** is
  what the model predicted for them, on average; **came back** is the share who actually did. Close
  numbers mean probabilities you can trust. A tree gives only a few different numbers, so some
  groups merge and you may see fewer than ten rows.

**Answer in a sentence:** which model would you hand the hospital, and which two numbers decide it?

### 3. How stable is a tree's story?

Section 5 grew the same tree on two samples and got two different stories. Here you count how often
that happens.

- **(a)** Draw 20 samples of 1,000 training discharges *with replacement* —
  `rng.integers(0, len(train), 1_000)` with `rng = np.random.default_rng(0)` — and fit
  `tree_model(max_depth=2)` to each. For each tree, record its first question and its two second
  questions with `first_questions(fitted)` (setup cell below). The pattern, with one blank for you:

  ```python
  rng = np.random.default_rng(0)                 # make it ONCE, before the loop
  first, second = Counter(), Counter()           # tallies: question -> how many trees asked it
  for _ in range(20):
      rows = rng.integers(0, len(train), 1_000)  # 1,000 random training rows; repeats allowed
      fitted = ...                               # tree_model(max_depth=2), fit to X_train.iloc[rows], y_train[rows]
      q1, q2 = first_questions(fitted)           # its first question, and a list of its second ones
      first[q1] += 1
      second.update(q2)
  print(first)
  print(second)
  ```

- **(b)** Tally them: `Counter(...)` does it above. How many different first questions? How many
  different second questions? (`len(first)`, `len(second)`.)
- **(c)** The same with samples of 5,000: change `1_000` to `5_000` and run it again.

**Answer in a sentence:** if a hospital read **one** tree as an explanation of who comes back, what
would you warn them about — with your tallies as the evidence?

In [ ]:
# Studio setup -- run this before you start.
hand = pd.DataFrame({
    "prior_stays": [0, 0, 0, 0, 1, 1, 1, 2, 3, 4],
    "emergency":   [0, 1, 0, 1, 1, 0, 1, 1, 0, 1],
    "age_70_plus": [0, 0, 1, 1, 0, 1, 1, 0, 1, 1],
    "came_back":   [0, 0, 0, 0, 0, 1, 0, 1, 1, 1],
}, index=range(1, 11))


def first_questions(fitted):
    """The column a fitted two-level tree pipeline asks about first, and the columns of its two
    second questions (a list: one or two names)."""
    t, names = fitted[-1].tree_, fitted[0].get_feature_names_out()
    second = [names[t.feature[c]] for c in (t.children_left[0], t.children_right[0]) if t.feature[c] >= 0]
    return names[t.feature[0]], second


print(hand)

In [ ]:
# YOUR CODE HERE
# my guess, before computing:

In [ ]:
# YOUR CODE HERE
# my guess, before computing:

In [ ]:
# YOUR CODE HERE
# my guess, before computing:

## Compare — at the start of class on Thursday

We compare answers at the start of Thursday's class, built from the notebooks you upload. So leave
these in your notebook, in words or as printed numbers:

1. Your guesses, written before you computed, next to what you found.
2. Question 1: your four split scores, and the question you picked.
3. Question 2: the tuned tree's and the logistic model's AUC and Brier score on test.

## Exit ticket

> Name one thing a single tree does **better** than Meeting 9's logistic regression and one thing it
> does **worse** — each with a number from today.

**Answer in the cell below, then save this notebook and upload it to Canvas under "Studio
notebook — Day 15". Your two studio points need this answered: a blank exit ticket earns zero.
It is not graded for being right — it is how I find out what landed.**

### Your exit ticket

*(double-click to edit, and replace this line)*

## Also in this neighborhood

📗 **ISLP §8.1** — this week's reading: regression and classification trees, cost-complexity
pruning (§8.1.1), and trees against linear models (§8.1.3).

📗 **ESL §9.2** — the same in more depth: why Gini and entropy, not the error rate, choose the splits.

📗 **Breiman, Friedman, Olshen & Stone (1984), *Classification and Regression Trees*** — CART, the
algorithm scikit-learn's `DecisionTreeClassifier` follows.

🎓 **Thursday:** bagging and random forests — many unstable trees, averaged.